# Segmentasi Transaksi & Deteksi Anomali

Alur pemodelan pada data transaksi bank sintetis: segmentasi nasabah dengan **K-Means**, penanda transaksi tidak biasa dengan **Isolation Forest**, dan aturan risiko login berulang.

**Sumber data:** dataset latihan Dicoding (adaptasi Kaggle *Bank Transaction Dataset for Fraud Detection*), transaksi kota-kota AS dalam USD tanpa label fraud asli.

**Cara pakai:** jalankan semua sel berurutan (*Run All*). Sel terakhir menyimpan data bersih dan artefak model untuk `app.py`. Karena tidak ada label fraud asli, seluruh keluaran merupakan kandidat tinjauan, dan bagian 4.1 serta 5.1 sampai 5.4 menguji kestabilannya.

---
# 1. Import Library & Muat Data

In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import spearmanr
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
sns.set_theme(style='whitegrid')


def show3(frame):
    """Tampilkan tabel dengan 3 desimal (untuk Jaccard, Spearman, ARI)."""
    with pd.option_context('display.float_format', '{:.3f}'.format):
        display(frame)

In [ ]:
# Pastikan path tetap benar meski notebook dijalankan dari folder notebooks/ atau root proyek
ROOT_DIR = (
    Path.cwd().parent
    if not (Path.cwd() / 'data').is_dir() and (Path.cwd().parent / 'data').is_dir()
    else Path.cwd()
)
local_candidates = [ROOT_DIR / 'data' / 'raw.xlsx', Path('raw.xlsx')]
local_path = next((p for p in local_candidates if p.exists()), None)
backup_url = 'https://docs.google.com/spreadsheets/d/e/2PACX-1vTbg5WVW6W3c8SPNUGc3A3AL-AG32TPEQGpdzARfNICMsLFI0LQj0jporhsLCeVhkN5AoRsTkn08AYl/pub?output=csv'
EXPECTED_SHAPE = (2537, 16)

if local_path:
    df_raw = pd.read_excel(local_path)
    print(f"Data dimuat dari: {local_path.name}")
else:
    df_raw = pd.read_csv(backup_url)
    print("Data dimuat dari URL cadangan.")

# URL cadangan bisa menyimpang diam-diam dari raw.xlsx, jadi ukuran data dicek
if df_raw.shape != EXPECTED_SHAPE:
    raise ValueError(f"Ukuran data {df_raw.shape} berbeda dari yang diharapkan {EXPECTED_SHAPE}. Periksa sumber data.")

print(f"Ukuran data: {df_raw.shape[0]} baris, {df_raw.shape[1]} kolom")
df_raw.head()

---
# 2. Pembersihan Data
1. Buang baris duplikat.
2. Buang kolom yang tidak dipakai: ID, IP address, dan kolom tanggal.
3. Isi nilai kosong: **median** untuk angka, **modus** untuk teks (data kosong kecil, sekitar 1% per kolom).

In [ ]:
df = df_raw.copy()

print(f"Baris duplikat: {df.duplicated().sum()}")
df = df.drop_duplicates().reset_index(drop=True)

df = df.drop(columns=[
    'TransactionID', 'AccountID', 'DeviceID', 'MerchantID',
    'IP Address', 'PreviousTransactionDate', 'TransactionDate',
])

print("\nData kosong per kolom (sebelum diisi):")
print(df.isnull().sum()[df.isnull().sum() > 0])

for col in df.select_dtypes(include='number').columns:
    df[col] = df[col].fillna(df[col].median())
for col in df.select_dtypes(include=['object', 'string']).columns:
    df[col] = df[col].fillna(df[col].mode()[0])

print(f"\nUkuran data bersih: {df.shape[0]} baris, {df.shape[1]} kolom")
print(f"Sisa data kosong: {df.isnull().sum().sum()}")

---
# 3. Fitur Tambahan & Eksplorasi Singkat
- **`Balance_Ratio`**: `TransactionAmount / AccountBalance`, seberapa besar porsi saldo yang dipakai.
- **`Amount_Per_Second`**: `TransactionAmount / TransactionDuration`, kecepatan nominal transaksi.
- **`High_Login_Risk`**: aturan `LoginAttempts >= 3`. Ini **aturan**, bukan hasil model.

In [ ]:
df['Balance_Ratio'] = df['TransactionAmount'] / df['AccountBalance']
df['Amount_Per_Second'] = df['TransactionAmount'] / df['TransactionDuration'].clip(lower=1)
df['High_Login_Risk'] = df['LoginAttempts'] >= 3

numeric_cols = ['TransactionAmount', 'CustomerAge', 'TransactionDuration',
                'LoginAttempts', 'AccountBalance', 'Balance_Ratio', 'Amount_Per_Second']

print(f"Transaksi dengan LoginAttempts >= 3: {int(df['High_Login_Risk'].sum())} ({df['High_Login_Risk'].mean() * 100:.2f}%)\n")
display(df[numeric_cols].describe().T)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.flatten(), numeric_cols):
    sns.histplot(df[col], bins=25, color='steelblue', ax=ax)
    ax.set_title(col, fontweight='bold')
axes.flatten()[-1].axis('off')
plt.suptitle('Sebaran Fitur Numerik', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

---
# 4. Segmentasi dengan K-Means
Aturan yang dipakai:
- **5 fitur perilaku saja**: `TransactionAmount`, `CustomerAge`, `TransactionDuration`, `AccountBalance`, `Balance_Ratio`. `LoginAttempts` sengaja tidak dipakai, supaya cluster tidak sekadar meniru aturan login.
- **`Balance_Ratio` dihitung dari nominal dan saldo**, jadi dua fitur itu ikut terhitung dua kali. Bagian 4.1 menguji seberapa besar pengaruhnya.
- **`StandardScaler`** dulu, supaya `AccountBalance` (ribuan USD) tidak mendominasi fitur lain.
- **Jumlah cluster** dipilih dengan Elbow dan Silhouette Score untuk `k = 2` sampai `8`.

In [ ]:
clustering_features = ['TransactionAmount', 'CustomerAge', 'TransactionDuration',
                       'AccountBalance', 'Balance_Ratio']

scaler = StandardScaler()
df_scaled = pd.DataFrame(scaler.fit_transform(df[clustering_features]), columns=clustering_features)

rows = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(df_scaled)
    rows.append({'k': k, 'Inertia': km.inertia_, 'Silhouette Score': silhouette_score(df_scaled, labels)})

eval_df = pd.DataFrame(rows)
show3(eval_df)

best_k = int(eval_df.loc[eval_df['Silhouette Score'].idxmax(), 'k'])
print(f"k terbaik menurut Silhouette Score: {best_k}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(eval_df['k'], eval_df['Inertia'], marker='o')
axes[0].set_title('Elbow (Inertia)')
axes[1].plot(eval_df['k'], eval_df['Silhouette Score'], marker='s', color='seagreen')
axes[1].set_title('Silhouette Score')
for ax in axes:
    ax.set_xlabel('k')
plt.tight_layout()
plt.show()

In [ ]:
# Silhouette tertinggi pada k = 2, tetapi selisihnya dengan k lain kecil (sekitar 0.23 sampai 0.26)
optimal_k = best_k
assert optimal_k == 2, "Penamaan segmen di bawah mengasumsikan k = 2"

kmeans_model = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
df['Cluster'] = kmeans_model.fit_predict(df_scaled)

# Nama segmen hanya menggambarkan yang terukur (usia dan saldo), diurutkan dari rata-rata usia
SEG_YOUNG = 'Nasabah Muda Saldo Kecil'
SEG_OLDER = 'Nasabah Lebih Tua Saldo Besar'
age_by_cluster = df.groupby('Cluster')['CustomerAge'].mean().sort_values()
name_map = {
    age_by_cluster.index[0]: SEG_YOUNG,
    age_by_cluster.index[1]: SEG_OLDER,
}
df['Segment_Name'] = df['Cluster'].map(name_map)

balance_by_segment = df.groupby('Segment_Name')['AccountBalance'].mean()
assert balance_by_segment[SEG_OLDER] > balance_by_segment[SEG_YOUNG], "Nama segmen tidak cocok dengan saldo rata-rata"

print("Jumlah per segmen:")
display(df['Segment_Name'].value_counts().to_frame('Jumlah').assign(**{'Persen (%)': lambda d: d['Jumlah'] / len(df) * 100}))

print("\nRata-rata fitur per segmen:")
display(df.groupby('Segment_Name')[clustering_features + ['LoginAttempts']].mean())

print("\nSegmen vs aturan risiko login:")
display(pd.crosstab(df['Segment_Name'], df['High_Login_Risk']))

**Catatan:** Silhouette Score di bawah 0.5 menunjukkan kedua segmen saling tumpang tindih. K-Means memisahkan **nasabah muda bersaldo kecil** dan **nasabah lebih tua bersaldo besar** sebagai gambaran umum, bukan batas tegas. Selisih `LoginAttempts` dan `TransactionDuration` antarsegmen mendekati nol, sehingga nama segmen hanya memuat usia dan saldo.

## 4.1 Seberapa Kokoh Segmentasinya?
Ulangi K-Means (k = 2) dengan konfigurasi fitur lain, lalu ukur kemiripan pembagiannya dengan segmentasi asli memakai *Adjusted Rand Index* (ARI, 1.0 = identik).

In [ ]:
variants = {
    'Asli (5 fitur)': (clustering_features, None),
    'log1p pada 5 fitur': (clustering_features, np.log1p),
    'Tanpa Balance_Ratio': (['TransactionAmount', 'CustomerAge', 'TransactionDuration', 'AccountBalance'], None),
    'Hanya usia + saldo': (['CustomerAge', 'AccountBalance'], None),
}

rows = []
for name, (feats, transform) in variants.items():
    X = df[feats] if transform is None else transform(df[feats])
    X = StandardScaler().fit_transform(X)
    labels = KMeans(n_clusters=2, random_state=42, n_init=10).fit_predict(X)
    rows.append({
        'Konfigurasi': name,
        'Silhouette': silhouette_score(X, labels),
        'ARI vs asli': adjusted_rand_score(df['Cluster'], labels),
    })
show3(pd.DataFrame(rows))

---
# 5. Deteksi Anomali dengan Isolation Forest
**Isolation Forest** memisahkan transaksi yang berbeda jauh dari mayoritas pola transaksi lain:

- **Fitur**: 5 fitur perilaku + `Amount_Per_Second`. `LoginAttempts` tidak dipakai agar hasil model dan aturan login berdiri sendiri.
- **`contamination = 0.05`**: parameter untuk menandai sekitar 5% transaksi, bukan temuan alami dari data.
- **`Anomaly_Raw`**: skor mentah model (semakin besar semakin tidak biasa; di atas 0 berarti anomali).
- **`Anomaly_Score`**: persentil 0-100 dari skor mentah pada data latih (skor 97 berarti lebih tidak biasa dari 97% data latih). Skala min-max dihindari karena sensitif terhadap satu outlier ekstrem.
- Hasilnya adalah **kandidat tinjauan**, bukan bukti fraud karena data tidak memiliki label fraud asli.

In [ ]:
CONTAMINATION = 0.05
anomaly_features = clustering_features + ['Amount_Per_Second']

scaler_anom = StandardScaler()
X_anom = scaler_anom.fit_transform(df[anomaly_features])

iso = IsolationForest(n_estimators=200, contamination=CONTAMINATION, random_state=42)
iso.fit(X_anom)

df['Anomaly_Flag'] = iso.predict(X_anom) == -1

# Skor mentah: makin besar makin tidak biasa. Anomali bila nilainya di atas 0.
df['Anomaly_Raw'] = -iso.decision_function(X_anom)
train_scores = np.sort(df['Anomaly_Raw'].to_numpy())


def to_percentile(raw, reference):
    """Persentil 0-100: persen data latih yang skor mentahnya tidak lebih besar dari `raw`."""
    return np.searchsorted(reference, raw, side='right') / len(reference) * 100


df['Anomaly_Score'] = to_percentile(df['Anomaly_Raw'].to_numpy(), train_scores)
anomaly_threshold = float(to_percentile(0.0, train_scores))

print(f"Transaksi ditandai anomali: {int(df['Anomaly_Flag'].sum())} ({df['Anomaly_Flag'].mean() * 100:.2f}%)")
print(f"Ambang anomali pada skala persentil: {anomaly_threshold:.2f}")
assert np.array_equal(df['Anomaly_Score'] > anomaly_threshold, df['Anomaly_Flag'].to_numpy()), "Flag dan ambang persentil tidak cocok"

print("\nRata-rata fitur: anomali vs bukan anomali")
display(df.groupby('Anomaly_Flag')[anomaly_features + ['LoginAttempts']].mean().round(2))

print("\n10 transaksi dengan skor tertinggi:")
display(df.sort_values('Anomaly_Raw', ascending=False)[
    ['TransactionAmount', 'AccountBalance', 'Balance_Ratio', 'TransactionDuration', 'Anomaly_Score']
].head(10))

## 5.1 Seberapa Stabil Hasilnya?
Tiga pengujian diukur dengan *Jaccard index* pada himpunan transaksi yang ditandai (1.0 = identik):

1. **`contamination`** (0.03, 0.05, 0.07): mengubah ambang pada skor yang sama sehingga himpunan bersarang penuh dan Jaccard mencerminkan rasio jumlah.
2. **Seed acak** (10 seed berbeda, fitur sama): menguji kestabilan urutan dan daftar anomali terhadap keacakan model.
3. **Ablasi fitur dan pembanding sederhana**: menghapus satu fitur turunan, atau membandingkan dengan aturan nilai terbesar satu fitur.

In [ ]:
base_flags = df['Anomaly_Flag'].to_numpy()
base_raw = df['Anomaly_Raw'].to_numpy()
n_flag = int(base_flags.sum())


def jaccard(a, b):
    return (a & b).sum() / (a | b).sum()


# 1) contamination
rows = []
for c in [0.03, 0.05, 0.07]:
    flags = IsolationForest(n_estimators=200, contamination=c, random_state=42).fit(X_anom).predict(X_anom) == -1
    smaller, larger = (flags, base_flags) if flags.sum() <= base_flags.sum() else (base_flags, flags)
    rows.append({
        'contamination': c,
        'Jumlah anomali': int(flags.sum()),
        'Bersarang dengan 0.05': bool((smaller & larger).sum() == smaller.sum()),
        'Jaccard vs 0.05': jaccard(flags, base_flags),
    })
show3(pd.DataFrame(rows))

In [ ]:
def fit_if(features, seed=42):
    X = StandardScaler().fit_transform(df[features])
    model = IsolationForest(n_estimators=200, contamination=CONTAMINATION, random_state=seed).fit(X)
    return -model.decision_function(X), model.predict(X) == -1


# 2) seed acak
rows = []
for seed in range(10):
    raw, flags = fit_if(anomaly_features, seed)
    rows.append({'Spearman skor': spearmanr(base_raw, raw)[0], 'Jaccard flag': jaccard(flags, base_flags)})
print("10 seed berbeda vs seed 42:")
show3(pd.DataFrame(rows).agg(['mean', 'min', 'max']))

# 3) ablasi fitur dan pembanding satu fitur
rows = []
for name, feats in {
    'Tanpa Amount_Per_Second': [f for f in anomaly_features if f != 'Amount_Per_Second'],
    'Tanpa Balance_Ratio': [f for f in anomaly_features if f != 'Balance_Ratio'],
    'Hanya nominal + saldo': ['TransactionAmount', 'AccountBalance'],
}.items():
    raw, flags = fit_if(feats)
    rows.append({'Konfigurasi': name, 'Spearman skor': spearmanr(base_raw, raw)[0], 'Jaccard flag': jaccard(flags, base_flags)})
for col in ['TransactionAmount', 'Balance_Ratio', 'Amount_Per_Second']:
    top = np.zeros(len(df), dtype=bool)
    top[df[col].nlargest(n_flag).index] = True
    rows.append({'Konfigurasi': f'Pembanding: {n_flag} nilai terbesar {col}', 'Spearman skor': spearmanr(base_raw, df[col])[0], 'Jaccard flag': jaccard(top, base_flags)})
print("\nAblasi fitur dan pembanding sederhana:")
show3(pd.DataFrame(rows))

## 5.2 Apa yang Menggerakkan Skor?
Korelasi peringkat Spearman antara skor mentah dan tiap fitur, diikuti tingkat anomali antarsegmen **di dalam kuintil `Balance_Ratio` yang sama**. Jika selisih antarsegmen hilang setelah rasio dikendalikan, perbedaan berasal dari definisi rasio (saldo kecil memperbesar rasio), bukan perilaku segmen.

In [ ]:
corr_features = anomaly_features + ['LoginAttempts']
show3(df[['Anomaly_Raw'] + corr_features].corr(method='spearman')['Anomaly_Raw'].drop('Anomaly_Raw').to_frame('Spearman dengan skor mentah'))

quintile = pd.qcut(df['Balance_Ratio'], 5, labels=['Q1 (terendah)', 'Q2', 'Q3', 'Q4', 'Q5 (tertinggi)'])
print("\nTingkat anomali (%) per segmen, keseluruhan:")
display(df.groupby('Segment_Name')['Anomaly_Flag'].mean().mul(100).round(2).to_frame('Anomali (%)'))
print("Tingkat anomali (%) per segmen di dalam kuintil Balance_Ratio:")
display(df.groupby([quintile, 'Segment_Name'], observed=True)['Anomaly_Flag'].mean().unstack().mul(100).round(1))

## 5.3 Irisan dengan Aturan Login
Kalau dua mekanisme itu saling independen, irisannya diharapkan sekitar `jumlah anomali x jumlah risiko login / total`.

In [ ]:
print("Sebaran LoginAttempts:")
display(df['LoginAttempts'].value_counts().sort_index().to_frame('Jumlah'))

expected = df['Anomaly_Flag'].sum() * df['High_Login_Risk'].sum() / len(df)
observed = int((df['Anomaly_Flag'] & df['High_Login_Risk']).sum())
print(f"Irisan teramati: {observed}, diharapkan jika independen: {expected:.1f}")

print("\nAnomali vs risiko login:")
display(pd.crosstab(df['High_Login_Risk'], df['Anomaly_Flag']))

## 5.4 Uji dengan Anomali Suntikan
Tanpa label fraud asli, 2% transaksi dipilih acak dan nominalnya dikali 5 (fitur turunan dihitung ulang). Model dilatih ulang tanpa mengetahui label suntikan, lalu *recall* diukur pada 5% skor teratas. Pengujian ini hanya memeriksa satu skenario sintetis, bukan akurasi fraud nyata.

In [ ]:
rng = np.random.default_rng(42)
inj = df[['TransactionAmount', 'CustomerAge', 'TransactionDuration', 'AccountBalance']].copy()
inj_idx = rng.choice(len(inj), size=int(0.02 * len(inj)), replace=False)
inj.loc[inj_idx, 'TransactionAmount'] *= 5
inj['Balance_Ratio'] = inj['TransactionAmount'] / inj['AccountBalance']
inj['Amount_Per_Second'] = inj['TransactionAmount'] / inj['TransactionDuration'].clip(lower=1)

is_inj = np.zeros(len(inj), dtype=bool)
is_inj[inj_idx] = True

X_inj = StandardScaler().fit_transform(inj[anomaly_features])
flagged = IsolationForest(n_estimators=200, contamination=CONTAMINATION, random_state=42).fit(X_inj).predict(X_inj) == -1
n_top = int(flagged.sum())

rows = [{'Metode': 'Isolation Forest', 'Recall suntikan': flagged[is_inj].mean()}]
for col in ['TransactionAmount', 'Balance_Ratio']:
    top = np.zeros(len(inj), dtype=bool)
    top[inj[col].nlargest(n_top).index] = True
    rows.append({'Metode': f'Pembanding: {n_top} nilai terbesar {col}', 'Recall suntikan': top[is_inj].mean()})

print(f"Transaksi disuntik: {int(is_inj.sum())}, ditandai per metode: {n_top}")
show3(pd.DataFrame(rows))

---
# 6. Simpan Hasil untuk Dashboard
File yang dibaca `app.py`:
- `data/data_clustering_clean.csv`: data lengkap dengan segmen, flag anomali, skor mentah, dan skor persentil
- `models/scaler_clustering.joblib`, `models/kmeans_model.joblib`
- `models/anomaly_model.joblib`: scaler + Isolation Forest + skor mentah data latih (dasar skor persentil)
- `models/k_evaluation.csv`: hasil Elbow/Silhouette

In [ ]:
data_dir = ROOT_DIR / 'data'
models_dir = ROOT_DIR / 'models'


def out_path(folder_dir, name):
    return folder_dir / name if folder_dir.is_dir() else Path(name)


anomaly_bundle = {
    'features': anomaly_features,
    'scaler': scaler_anom,
    'model': iso,
    'train_scores': train_scores,  # skor mentah data latih, terurut: dasar skor persentil di app
    'contamination': CONTAMINATION,
}

df.to_csv(out_path(data_dir, 'data_clustering_clean.csv'), index=False)
joblib.dump(scaler, out_path(models_dir, 'scaler_clustering.joblib'))
joblib.dump(kmeans_model, out_path(models_dir, 'kmeans_model.joblib'))
joblib.dump(anomaly_bundle, out_path(models_dir, 'anomaly_model.joblib'))
eval_df.to_csv(out_path(models_dir, 'k_evaluation.csv'), index=False)

print("File tersimpan.\n")
print("Ringkasan metrik:")
print(f"Total transaksi          : {len(df):,}")
print(f"Silhouette (k = 2)       : {eval_df.loc[eval_df['k'] == 2, 'Silhouette Score'].iloc[0]:.4f}")
for name, n in df['Segment_Name'].value_counts().items():
    print(f"Segmen {name:<30}: {n:,} ({n / len(df) * 100:.2f}%)")
print(f"Anomali (Isolation Forest): {int(df['Anomaly_Flag'].sum())} ({df['Anomaly_Flag'].mean() * 100:.2f}%)")
print(f"Risiko login (>= 3)      : {int(df['High_Login_Risk'].sum())} ({df['High_Login_Risk'].mean() * 100:.2f}%)")